# Lab 06 External V2 — 04 Gold Aggregations

**Dataset:** Synthea Healthcare  
**Architecture:** External Delta tables  
**Compute:** Databricks Serverless compatible

## Purpose

Create four dashboard-ready external Delta aggregates from the validated Gold facts:
`agg_daily_encounters`, `agg_organization_performance`,
`agg_payer_performance`, and `agg_condition_summary`.

> Serverless compatibility rule: this notebook does **not** call
> `REFRESH TABLE`, `CACHE TABLE`, `UNCACHE TABLE`, or Spark cache-refresh APIs.


## 1. Runtime context

In [ ]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "REPLACE_WITH_EXTERNAL_GOLD_ROOT",
    "05 External Gold root",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "06 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

if not external_gold_root or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT":
    raise ValueError(
        "external_gold_root must be passed by lab06_00_dev_runner "
        "or entered manually."
    )

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError("external_gold_root must be an abfss:// path.")

source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
source_csv_path = f"{source_volume_path}/source/csv"
reference_path = f"{source_volume_path}/reference"
target_schema_fqn = f"{catalog}.{target_schema}"

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Run validation     : {run_validation}")

## 2. Helpers and required objects

In [ ]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.external_tables import (
    normalize_location,
    overwrite_external_delta,
    registered_table_location,
    register_external_delta_table,
    validate_registered_location,
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}")

print("Serverless-safe external-table helpers loaded.")

TABLES = {
    "fact_encounters": f"{target_schema_fqn}.fact_encounters",
    "fact_conditions": f"{target_schema_fqn}.fact_conditions",
    "dim_organization": f"{target_schema_fqn}.dim_organization",
    "dim_payer": f"{target_schema_fqn}.dim_payer",
    "dim_condition": f"{target_schema_fqn}.dim_condition",
    "agg_daily_encounters": f"{target_schema_fqn}.agg_daily_encounters",
    "agg_organization_performance": f"{target_schema_fqn}.agg_organization_performance",
    "agg_payer_performance": f"{target_schema_fqn}.agg_payer_performance",
    "agg_condition_summary": f"{target_schema_fqn}.agg_condition_summary",
}

LOCATIONS = {
    name: f"{external_gold_root}/{name}"
    for name in [
        "agg_daily_encounters",
        "agg_organization_performance",
        "agg_payer_performance",
        "agg_condition_summary",
    ]
}

for required in [
    "fact_encounters","fact_conditions","dim_organization","dim_payer","dim_condition"
]:
    if not spark.catalog.tableExists(TABLES[required]):
        raise RuntimeError(f"Missing required Gold object: {TABLES[required]}")

fact_encounters = spark.table(TABLES["fact_encounters"])
fact_conditions = spark.table(TABLES["fact_conditions"])
dim_organization = spark.table(TABLES["dim_organization"])
dim_payer = spark.table(TABLES["dim_payer"])
dim_condition = spark.table(TABLES["dim_condition"])

print(f"fact_encounters: {fact_encounters.count():,}")
print(f"fact_conditions: {fact_conditions.count():,}")

## 3. `agg_daily_encounters`

In [ ]:
agg_daily_encounters_df = (
    fact_encounters
    .groupBy("date_key","encounter_date")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.countDistinct("organization_key").alias("organizations_active"),
        F.countDistinct("provider_key").alias("providers_active"),
        F.round(F.avg("duration_minutes"),2).alias("avg_duration_minutes"),
        F.round(F.sum("base_encounter_cost"),2).alias("base_encounter_cost"),
        F.round(F.sum("total_claim_cost"),2).alias("total_claim_cost"),
        F.round(F.sum("payer_coverage"),2).alias("payer_coverage"),
        F.round(F.sum("patient_responsibility"),2).alias("patient_responsibility"),
        F.sum(
            F.when(F.lower("encounter_class") == "emergency",1).otherwise(0)
        ).alias("emergency_encounters"),
    )
    .withColumn(
        "emergency_encounter_pct",
        F.round(
            F.col("emergency_encounters") / F.col("encounter_count") * F.lit(100.0),
            2,
        ),
    )
)

overwrite_external_delta(
    spark, agg_daily_encounters_df,
    TABLES["agg_daily_encounters"],
    LOCATIONS["agg_daily_encounters"],
)

display(spark.table(TABLES["agg_daily_encounters"]).orderBy("encounter_date").limit(20))

## 4. `agg_organization_performance`

In [ ]:
agg_organization_performance_df = (
    fact_encounters
    .groupBy("organization_key")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.countDistinct("provider_key").alias("unique_providers"),
        F.round(F.avg("duration_minutes"),2).alias("avg_duration_minutes"),
        F.round(F.sum("total_claim_cost"),2).alias("total_claim_cost"),
        F.round(F.avg("total_claim_cost"),2).alias("avg_claim_cost"),
        F.round(F.sum("payer_coverage"),2).alias("payer_coverage"),
        F.round(F.sum("patient_responsibility"),2).alias("patient_responsibility"),
        F.min("encounter_date").alias("first_encounter_date"),
        F.max("encounter_date").alias("last_encounter_date"),
    )
    .join(
        dim_organization.select(
            "organization_key","organization_id","organization_name","city","state"
        ),
        "organization_key",
        "left",
    )
    .select(
        "organization_key","organization_id","organization_name","city","state",
        "encounter_count","unique_patients","unique_providers",
        "avg_duration_minutes","total_claim_cost","avg_claim_cost",
        "payer_coverage","patient_responsibility",
        "first_encounter_date","last_encounter_date",
    )
)

overwrite_external_delta(
    spark, agg_organization_performance_df,
    TABLES["agg_organization_performance"],
    LOCATIONS["agg_organization_performance"],
)

display(
    spark.table(TABLES["agg_organization_performance"])
    .orderBy(F.desc("encounter_count"))
    .limit(20)
)

## 5. `agg_payer_performance`

In [ ]:
agg_payer_performance_df = (
    fact_encounters
    .groupBy("payer_key")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.round(F.sum("total_claim_cost"),2).alias("total_claim_cost"),
        F.round(F.avg("total_claim_cost"),2).alias("avg_claim_cost"),
        F.round(F.sum("payer_coverage"),2).alias("payer_coverage"),
        F.round(F.sum("patient_responsibility"),2).alias("patient_responsibility"),
    )
    .withColumn(
        "coverage_pct",
        F.when(
            F.col("total_claim_cost") > 0,
            F.round(F.col("payer_coverage") / F.col("total_claim_cost") * 100.0, 2),
        ).otherwise(F.lit(0.0)),
    )
    .join(
        dim_payer.select("payer_key","payer_id","payer_name"),
        "payer_key",
        "left",
    )
    .select(
        "payer_key","payer_id","payer_name",
        "encounter_count","unique_patients",
        "total_claim_cost","avg_claim_cost",
        "payer_coverage","patient_responsibility","coverage_pct",
    )
)

overwrite_external_delta(
    spark, agg_payer_performance_df,
    TABLES["agg_payer_performance"],
    LOCATIONS["agg_payer_performance"],
)

display(
    spark.table(TABLES["agg_payer_performance"])
    .orderBy(F.desc("total_claim_cost"))
)

## 6. `agg_condition_summary`

In [ ]:
agg_condition_summary_df = (
    fact_conditions
    .groupBy("condition_key")
    .agg(
        F.count("*").alias("condition_events"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.sum(F.when(F.col("is_active_condition"),1).otherwise(0)).alias("active_condition_events"),
        F.round(F.avg("condition_duration_days"),2).alias("avg_duration_days"),
        F.min("condition_start_date").alias("first_observed_date"),
        F.max("condition_start_date").alias("last_observed_date"),
    )
    .join(
        dim_condition.select(
            "condition_key","condition_code","condition_description"
        ),
        "condition_key",
        "left",
    )
    .select(
        "condition_key","condition_code","condition_description",
        "condition_events","unique_patients","active_condition_events",
        "avg_duration_days","first_observed_date","last_observed_date",
    )
)

overwrite_external_delta(
    spark, agg_condition_summary_df,
    TABLES["agg_condition_summary"],
    LOCATIONS["agg_condition_summary"],
)

display(
    spark.table(TABLES["agg_condition_summary"])
    .orderBy(F.desc("condition_events"))
    .limit(25)
)

## 7. Grain, reconciliation, and external-location validation

In [ ]:
grain_checks = []

for name, key_cols in {
    "agg_daily_encounters": ["date_key"],
    "agg_organization_performance": ["organization_key"],
    "agg_payer_performance": ["payer_key"],
    "agg_condition_summary": ["condition_key"],
}.items():
    df = spark.table(TABLES[name])
    rows = df.count()
    duplicates = (
        df.groupBy(*key_cols).count().filter(F.col("count") > 1).count()
    )
    grain_checks.append((name, rows > 0 and duplicates == 0))

encounter_rows = fact_encounters.count()
daily_sum = (
    spark.table(TABLES["agg_daily_encounters"])
    .agg(F.sum("encounter_count").alias("n"))
    .first()["n"]
)
org_sum = (
    spark.table(TABLES["agg_organization_performance"])
    .agg(F.sum("encounter_count").alias("n"))
    .first()["n"]
)
payer_sum = (
    spark.table(TABLES["agg_payer_performance"])
    .agg(F.sum("encounter_count").alias("n"))
    .first()["n"]
)
condition_rows = fact_conditions.count()
condition_sum = (
    spark.table(TABLES["agg_condition_summary"])
    .agg(F.sum("condition_events").alias("n"))
    .first()["n"]
)

locations_ok = all(
    normalize_location(registered_table_location(spark, TABLES[name]))
    == normalize_location(LOCATIONS[name])
    for name in LOCATIONS
)

final_checks = (
    grain_checks
    + [
        ("daily_reconciliation", int(daily_sum or 0) == encounter_rows),
        ("organization_reconciliation", int(org_sum or 0) == encounter_rows),
        ("payer_reconciliation", int(payer_sum or 0) == encounter_rows),
        ("condition_reconciliation", int(condition_sum or 0) == condition_rows),
        ("external_locations", locations_ok),
    ]
)

display(spark.createDataFrame(
    [(n, "PASS" if ok else "FAIL") for n, ok in final_checks],
    ["validation_area","status"],
))

failed = [n for n, ok in final_checks if not ok]
if run_validation and failed:
    raise RuntimeError("04 Aggregations failed: " + ", ".join(failed))

print("LAB 06 EXTERNAL V2 — GOLD AGGREGATIONS COMPLETE")
print("Serverless compatibility: PASS")
print("REFRESH TABLE calls: 0")
print("Next: lab06_07_validation")